# Lab 10: Custom Architectures - Building Neural Networks from Scratch

## Lab Overview

This lab focuses on building custom neural network architectures using PyTorch. You'll learn to design, implement, and optimize custom models, including transformer variants and hybrid architectures for specific use cases.

## Learning Objectives

By the end of this lab, you will:
- Master PyTorch Module system for custom architectures
- Learn to design efficient network architectures
- Understand model composition and modularity
- Explore architectural innovations in transformers
- Set up AMD GPU backend for custom model training
- Implement and test novel architecture designs

---

## Step 1: Setup and AMD GPU Configuration

We'll configure the AMD RyzenAI backend for efficient custom architecture development and training. This ensures optimal performance for complex model architectures on AMD hardware.

**Key Libraries:**
- `torch`: Core PyTorch functionality with ROCm support
- `torch.nn`: Neural network modules and layers
- `transformers`: Hugging Face transformers library
- `peft`: Parameter Efficient Fine-Tuning library
- `jupyter_gpu_init`: AMD GPU initialization for RyzenAI

In [ ]:
# Initialize AMD GPU Backend and Required Libraries
import sys
sys.path.append('../')



# Core libraries for custom architectures
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Transformer and PEFT libraries
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from peft import get_peft_model, LoraConfig, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    print("PEFT not available. Install with: pip install peft")
    PEFT_AVAILABLE = False

import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

print("Environment ready for custom architecture implementation")

AMD RyzenAI GPU environment initialized successfully


2025-10-13 03:54:53.618759: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda
PyTorch version: 2.6.0+gitdbfe118
GPU: AMD Radeon Graphics
GPU Memory: 68.7 GB
Environment ready for custom architecture implementation


## Step 2: PyTorch Module System Fundamentals

Understanding the `nn.Module` system is crucial for building custom architectures. This system provides the foundation for all neural network components in PyTorch.

**Core Module Concepts:**
- **Inheritance**: All custom models inherit from `nn.Module`
- **Parameter Registration**: Automatic tracking of learnable parameters
- **Forward Pass**: Define computation in the `forward()` method
- **State Management**: Model parameters, buffers, and training state

**Key Methods:**
- **`__init__()`**: Initialize layers and parameters
- **`forward()`**: Define the forward computation
- **`parameters()`**: Access all learnable parameters
- **`state_dict()`**: Get model state for saving/loading

**Device Management:**
- **`.to(device)`**: Move model to GPU/CPU
- **Automatic propagation**: All parameters move together
- **Consistency**: Ensure model and data are on same device

Let's build a custom CNN architecture and explore these concepts:

In [ ]:
# Custom CNN Architecture Implementation
class CustomCNN(nn.Module):
    """Custom Convolutional Neural Network for image classification"""

    def __init__(self, num_classes=10, dropout_rate=0.2):
        super(CustomCNN, self).__init__()

        # Store configuration
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        # Convolutional layers with batch normalization
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, padding=2)  # 32x32x3 → 32x32x32
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)  # 32x32x32 → 16x16x32

        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=2)  # 16x16x32 → 16x16x64
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)  # 16x16x64 → 8x8x64

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # 8x8x64 → 8x8x128
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)  # 8x8x128 → 4x4x128

        # Fully connected layers with dropout
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize model weights using Xavier/He initialization"""
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        """Forward pass through the network"""
        # Convolutional blocks with activation and normalization
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)  # Flatten: [batch, 128, 4, 4] → [batch, 2048]

        # Fully connected layers with dropout
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)  # No activation - raw logits

        return x

    def get_feature_maps(self, x):
        """Extract intermediate feature maps for visualization"""
        features = {}

        # Conv1 features
        x = F.relu(self.bn1(self.conv1(x)))
        features['conv1'] = x.clone()
        x = self.pool1(x)

        # Conv2 features
        x = F.relu(self.bn2(self.conv2(x)))
        features['conv2'] = x.clone()
        x = self.pool2(x)

        # Conv3 features
        x = F.relu(self.bn3(self.conv3(x)))
        features['conv3'] = x.clone()

        return features

# Create and analyze the model
print("Custom CNN Architecture:")
model = CustomCNN(num_classes=10).to(device)
print(f"Model created and moved to: {model.conv1.weight.device}")
print(f"\nModel architecture:\n{model}")

# Analyze model complexity
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Analysis:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (MB): {total_params * 4 / 1024**2:.2f}")  # Assuming float32

# Parameter breakdown by layer type
conv_params = sum(p.numel() for name, p in model.named_parameters() if 'conv' in name)
fc_params = sum(p.numel() for name, p in model.named_parameters() if 'fc' in name)
bn_params = sum(p.numel() for name, p in model.named_parameters() if 'bn' in name)

print(f"\nParameter distribution:")
print(f"  Convolutional layers: {conv_params:,} ({100*conv_params/total_params:.1f}%)")
print(f"  Fully connected layers: {fc_params:,} ({100*fc_params/total_params:.1f}%)")
print(f"  Batch normalization: {bn_params:,} ({100*bn_params/total_params:.1f}%)")

Custom CNN Architecture:
Model created and moved to: cuda:0

Model architecture:
CustomCNN(
  (conv1): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=2048, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features

## Step 3: Model Testing and State Management

Testing custom architectures and managing model state (saving/loading) are crucial skills for deep learning development.

**Model Testing:**
- **Forward Pass Validation**: Ensure correct input/output shapes
- **Device Consistency**: Verify model and data are on same device
- **Gradient Flow**: Check that gradients flow through all parameters
- **Memory Usage**: Monitor GPU memory consumption

**State Management:**
- **state_dict()**: Dictionary containing all learnable parameters
- **save/load**: Persistent storage of model weights
- **Device Transfer**: Proper handling when loading on different devices
- **Checkpoint Strategy**: Saving intermediate training states

**Best Practices:**
- Always test with dummy data first
- Verify shapes at each layer
- Monitor memory usage during development
- Use consistent device placement

In [ ]:
# Model Testing and Validation
print("Model Testing and Validation:")

# Test forward pass with dummy data
batch_size = 4
test_input = torch.randn(batch_size, 3, 32, 32, device=device)
print(f"Test input shape: {test_input.shape}")
print(f"Test input device: {test_input.device}")

# Forward pass (evaluation mode first)
model.eval()  # Set to evaluation mode
with torch.no_grad():
    output = model(test_input)

print(f"Model output shape: {output.shape}")
print(f"Expected shape: [batch_size={batch_size}, num_classes={model.num_classes}]")
print(f"Shape correct: {output.shape == (batch_size, model.num_classes)}")

# Test feature extraction
features = model.get_feature_maps(test_input[:1])  # Use single sample
print(f"\nFeature Maps Analysis:")
for layer_name, feature_map in features.items():
    print(f"  {layer_name}: {feature_map.shape}")

# Check gradient flow - IMPORTANT: Set model to training mode and enable gradients
print(f"\nGradient Flow Test:")
model.train()  # Set to training mode - CRITICAL for gradient computation
model.zero_grad()  # Clear any existing gradients

# Create new forward pass with gradients enabled
test_input_grad = torch.randn(batch_size, 3, 32, 32, device=device, requires_grad=True)
output_grad = model(test_input_grad)

# Compute dummy loss and backward pass
dummy_loss = output_grad.sum()
print(f"Dummy loss: {dummy_loss.item():.6f}")
print(f"Loss requires grad: {dummy_loss.requires_grad}")

# Perform backward pass
dummy_loss.backward()

grad_check = {}
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_check[name] = grad_norm
    else:
        print(f"No gradient for parameter: {name}")

print(f"Layers with gradients: {len(grad_check)}/{len(list(model.parameters()))}")
print(f"Sample gradient norms:")
for name, norm in list(grad_check.items())[:5]:  # Show first 5
    print(f"  {name}: {norm:.6f}")

# Verify gradients are flowing properly
if len(grad_check) == len(list(model.parameters())):
    print("All parameters have gradients - gradient flow is working correctly!")
else:
    print("Some parameters missing gradients - check model architecture")

# Model State Management
print(f"\nModel State Management:")

# Save model state
model_path = 'custom_cnn_weights.pt'
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

# Analyze state dictionary
state_dict = model.state_dict()
print(f"State dict keys: {len(state_dict)} tensors")
print(f"Sample keys: {list(state_dict.keys())[:5]}")

# Create new model and load weights
print(f"\nModel Loading Test:")
model_new = CustomCNN(num_classes=10).to(device)

# Compare weights before loading
original_weight = model.conv1.weight[0, 0, 0, 0].item()
new_weight_before = model_new.conv1.weight[0, 0, 0, 0].item()
print(f"Original model conv1 weight sample: {original_weight:.6f}")
print(f"New model conv1 weight sample (before loading): {new_weight_before:.6f}")

# Load saved weights
model_new.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
new_weight_after = model_new.conv1.weight[0, 0, 0, 0].item()
print(f"New model conv1 weight sample (after loading): {new_weight_after:.6f}")
print(f"Weights match: {abs(original_weight - new_weight_after) < 1e-6}")

# Verify models produce same output (use evaluation mode)
model.eval()
model_new.eval()
with torch.no_grad():
    output_original = model(test_input)
    output_loaded = model_new(test_input)

outputs_match = torch.allclose(output_original, output_loaded, atol=1e-6)
print(f"Model outputs match: {outputs_match}")

# Memory usage analysis
if device.type == 'cuda':
    memory_allocated = torch.cuda.memory_allocated(device) / 1024**2
    memory_reserved = torch.cuda.memory_reserved(device) / 1024**2
    print(f"\nGPU Memory Usage:")
    print(f"  Allocated: {memory_allocated:.2f} MB")
    print(f"  Reserved: {memory_reserved:.2f} MB")

print(f"\nModel testing completed successfully!")

Model Testing and Validation:
Test input shape: torch.Size([4, 3, 32, 32])
Test input device: cuda:0
Model output shape: torch.Size([4, 10])
Expected shape: [batch_size=4, num_classes=10]
Shape correct: True

Feature Maps Analysis:
  conv1: torch.Size([1, 32, 32, 32])
  conv2: torch.Size([1, 64, 16, 16])
  conv3: torch.Size([1, 128, 8, 8])

Gradient Flow Test:
Dummy loss: 23.797703
Loss requires grad: True
Layers with gradients: 18/18
Sample gradient norms:
  conv1.weight: 326.938232
  conv1.bias: 0.000013
  bn1.weight: 13.408196
  bn1.bias: 7.310006
  conv2.weight: 373.129242
All parameters have gradients - gradient flow is working correctly!

Model State Management:
Model saved to: custom_cnn_weights.pt
State dict keys: 27 tensors
Sample keys: ['conv1.weight', 'conv1.bias', 'bn1.weight', 'bn1.bias', 'bn1.running_mean']

Model Loading Test:
Original model conv1 weight sample: 0.036743
New model conv1 weight sample (before loading): 0.048091
New model conv1 weight sample (after loading

## Step 4: Transformer Integration and Parameter-Efficient Fine-Tuning

Now we'll explore working with pre-trained transformer models and applying parameter-efficient fine-tuning techniques like LoRA (Low-Rank Adaptation).

**Transformer Model Loading:**
- **Pre-trained Models**: Load models from Hugging Face Hub
- **Device Management**: Ensure models are properly placed on AMD GPU
- **Memory Optimization**: Handle large models efficiently
- **Model Architecture Analysis**: Understanding transformer structure

**Parameter-Efficient Fine-Tuning (PEFT):**
- **LoRA (Low-Rank Adaptation)**: Efficient fine-tuning with minimal parameters
- **Rank Decomposition**: Express weight updates as low-rank matrices
- **Memory Efficiency**: Reduce memory requirements for large model adaptation
- **Task-Specific Adaptation**: Customize models for specific use cases

**Key Concepts:**
- **Frozen Parameters**: Keep original model weights unchanged
- **Adapter Layers**: Add small trainable modules
- **Gradient Efficiency**: Only compute gradients for adapter parameters
- **Model Composition**: Combine multiple adapters for different tasks

**Note**: For demonstration, we'll use a small model path. In practice, you would use actual model identifiers like `"meta-llama/Llama-2-7b-hf"`.

In [ ]:
# Load Pre-trained Transformer Model with AMD GPU Support
try:
    from transformers import AutoModel, AutoTokenizer, AutoConfig
    from peft import get_peft_model, LoraConfig, TaskType

    print("Loading transformer model components...")

    # For demonstration, we'll create a minimal transformer-like model
    # In practice, you would use: model_name = "meta-llama/Llama-2-7b-hf"

    # Create a simple transformer configuration for demonstration
    config = AutoConfig.from_pretrained("gpt2")  # Using GPT-2 config as base
    config.num_hidden_layers = 2  # Smaller for demonstration
    config.num_attention_heads = 4
    config.hidden_size = 256

    print(f"Model configuration:")
    print(f"- Hidden size: {config.hidden_size}")
    print(f"- Number of layers: {config.num_hidden_layers}")
    print(f"- Attention heads: {config.num_attention_heads}")
    print(f"- Vocabulary size: {config.vocab_size}")

    # Create model from config (for demonstration)
    model = AutoModel.from_config(config)

    # Move model to AMD GPU
    model = model.to(device)
    print(f"Model moved to {device}")

    # Model analysis
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\nModel Analysis:")
    print(f"- Total parameters: {total_params:,}")
    print(f"- Trainable parameters: {trainable_params:,}")
    print(f"- Model device: {next(model.parameters()).device}")

except Exception as e:
    print(f"Error loading transformer model: {e}")
    print("Note: This is expected in demo environment. In practice, use actual model paths.")

Loading transformer model components...
Model configuration:
- Hidden size: 256
- Number of layers: 2
- Attention heads: 4
- Vocabulary size: 50257
Model moved to cuda

Model Analysis:
- Total parameters: 14,707,968
- Trainable parameters: 14,707,968
- Model device: cuda:0


In [ ]:
# Configure LoRA for Parameter-Efficient Fine-Tuning
try:
    print("Configuring LoRA (Low-Rank Adaptation)...")

    # LoRA configuration
    lora_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,  # Task type
        inference_mode=False,  # Training mode
        r=16,  # Rank of adaptation
        lora_alpha=32,  # LoRA scaling parameter
        lora_dropout=0.1,  # Dropout for LoRA layers
        target_modules=["c_attn", "c_proj"],  # Target modules for LoRA
        bias="none",  # How to handle bias parameters
    )

    print(f"LoRA Configuration:")
    print(f"- Rank (r): {lora_config.r}")
    print(f"- Alpha: {lora_config.lora_alpha}")
    print(f"- Dropout: {lora_config.lora_dropout}")
    print(f"- Target modules: {lora_config.target_modules}")

    # Apply LoRA to the model
    if 'model' in locals():
        peft_model = get_peft_model(model, lora_config)
        peft_model = peft_model.to(device)

        # Analyze parameter efficiency
        total_params = sum(p.numel() for p in peft_model.parameters())
        trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

        print(f"\nPEFT Model Analysis:")
        print(f"- Total parameters: {total_params:,}")
        print(f"- Trainable parameters: {trainable_params:,}")
        print(f"- Trainable ratio: {100 * trainable_params / total_params:.2f}%")
        print(f"- Memory reduction: {100 * (1 - trainable_params / total_params):.1f}%")

        # Test forward pass with AMD GPU
        print(f"\nTesting PEFT model forward pass...")
        # Fix: Create integer tensor for embedding layers (not float)
        test_input = torch.randint(0, config.vocab_size, (1, 10), dtype=torch.long).to(device)

        print(f"Test input shape: {test_input.shape}")
        print(f"Test input dtype: {test_input.dtype}")
        print(f"Test input device: {test_input.device}")
        print(f"Test input range: [{test_input.min().item()}, {test_input.max().item()}]")

        with torch.no_grad():
            outputs = peft_model(test_input)
            print(f"Forward pass successful!")
            print(f"- Input shape: {test_input.shape}")
            print(f"- Output shape: {outputs.last_hidden_state.shape}")
            print(f"- Output device: {outputs.last_hidden_state.device}")
            print(f"- Output dtype: {outputs.last_hidden_state.dtype}")

        print(f"\nKey Benefits of LoRA:")
        print(f"- Reduces trainable parameters by ~{100 * (1 - trainable_params / total_params):.1f}%")
        print(f"- Maintains model performance with minimal adaptation")
        print(f"- Enables efficient fine-tuning on AMD GPU")
        print(f"- Allows multiple task-specific adapters")

    else:
        print("Model not available for LoRA configuration")

except Exception as e:
    print(f"Error configuring LoRA: {e}")
    print("Note: This requires transformers and peft libraries to be installed")
    print(f"Error details: {type(e).__name__}: {str(e)}")

    # Fallback demonstration without PEFT
    print("\nFallback: Manual LoRA-like implementation...")

    class SimpleLoRALayer(nn.Module):
        """Simple LoRA-like layer for demonstration"""
        def __init__(self, in_features, out_features, rank=16):
            super().__init__()
            self.rank = rank
            self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
            self.lora_B = nn.Parameter(torch.randn(out_features, rank) * 0.01)
            self.scale = 1.0

        def forward(self, x):
            # LoRA: x @ (A.T @ B.T) = x @ (B @ A).T
            lora_weight = self.lora_B @ self.lora_A
            return F.linear(x, lora_weight * self.scale)

    # Demonstrate simple LoRA layer
    lora_demo = SimpleLoRALayer(256, 256, rank=16).to(device)
    test_tensor = torch.randn(1, 10, 256).to(device)

    with torch.no_grad():
        lora_output = lora_demo(test_tensor)
        print(f"Simple LoRA demonstration successful!")
        print(f"- Input shape: {test_tensor.shape}")
        print(f"- Output shape: {lora_output.shape}")
        print(f"- LoRA parameters: {sum(p.numel() for p in lora_demo.parameters()):,}")

Configuring LoRA (Low-Rank Adaptation)...
LoRA Configuration:
- Rank (r): 16
- Alpha: 32
- Dropout: 0.1
- Target modules: {'c_attn', 'c_proj'}

PEFT Model Analysis:
- Total parameters: 14,798,080
- Trainable parameters: 90,112
- Trainable ratio: 0.61%
- Memory reduction: 99.4%

Testing PEFT model forward pass...
Test input shape: torch.Size([1, 10])
Test input dtype: torch.int64
Test input device: cuda:0
Test input range: [3808, 50219]
Forward pass successful!
- Input shape: torch.Size([1, 10])
- Output shape: torch.Size([1, 10, 256])
- Output device: cuda:0
- Output dtype: torch.float32

Key Benefits of LoRA:
- Reduces trainable parameters by ~99.4%
- Maintains model performance with minimal adaptation
- Enables efficient fine-tuning on AMD GPU
- Allows multiple task-specific adapters


## Lab Summary and Key Learnings

### What We Accomplished

In this lab, we explored building custom neural network architectures from scratch while leveraging AMD RyzenAI GPU acceleration. We covered:

**1. Custom Architecture Design**
- Built a `CustomCNN` class with modern architectural components
- Implemented batch normalization for training stability
- Applied proper weight initialization techniques
- Designed modular, extensible network structures

**2. AMD GPU Integration**
- Configured PyTorch with ROCm support for AMD GPUs
- Implemented proper device management and tensor placement
- Optimized memory usage for custom architectures
- Validated GPU acceleration performance

**3. Model Analysis and Testing**
- Performed comprehensive forward pass validation
- Tested gradient flow and backpropagation
- Analyzed model parameters and computational requirements
- Implemented model state management (save/load)

**4. Advanced Integration Techniques**
- Explored transformer model loading with AMD GPU support
- Implemented Parameter-Efficient Fine-Tuning (PEFT) with LoRA
- Demonstrated memory-efficient adaptation strategies
- Connected custom architectures to modern transformer paradigms

### Next Steps and Advanced Topics

**Immediate Extensions:**
1. Implement more complex custom architectures (ResNet, DenseNet variants)
2. Explore advanced normalization techniques (LayerNorm, GroupNorm)
3. Integrate attention mechanisms into custom architectures
4. Experiment with different LoRA configurations and target modules

**Advanced Explorations:**
1. **Multi-GPU Training**: Scale custom architectures across multiple AMD GPUs
2. **Mixed Precision**: Implement FP16/BF16 training for memory efficiency
3. **Architecture Search**: Automated neural architecture search (NAS) techniques
4. **Quantization**: Post-training quantization for deployment optimization

**Integration Projects:**
1. Combine custom CNN features with transformer architectures
2. Build hybrid models for multimodal applications
3. Implement custom loss functions and training procedures
4. Create domain-specific architectures for specialized tasks

### Reflection Questions

1. How do architectural choices impact model performance and efficiency?
2. What are the trade-offs between custom architectures and pre-trained models?
3. How can parameter-efficient techniques reduce computational requirements?
4. What role does hardware-specific optimization play in modern deep learning?

---